# Domain-Specific Fine-Tuning of Qwen2.5-7B using LoRA for Conversation Summarization

**Objective:** Fine-tune Qwen2.5-7B with 4-bit QLoRA/LoRA on the SAMSum dialogue summarization dataset.

**Pipeline:** Qwen2.5-7B → 4-bit quantization → LoRA → 5,000 SAMSum examples → supervised fine-tuning → dialogue summarization.


In [1]:
!pip install -q unsloth
!pip install -q --upgrade transformers datasets trl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.15.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Dataset

SAMSum is used for dialogue summarization. A 5,000-example subset is used for the initial experiment so training remains practical on a Colab Tesla T4.


In [4]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum", split="train")
print(dataset)

dataset = dataset.shuffle(seed=3407).select(range(5000))
print("Number of training examples:", len(dataset))


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'dialogue', 'summary'],
    num_rows: 14731
})
Number of training examples: 5000


In [5]:
alpaca_prompt = """Below is a conversation between multiple people.

### Conversation:
{}

### Summary:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    dialogues = examples["dialogue"]
    summaries = examples["summary"]
    texts = []

    for dialogue, summary in zip(dialogues, summaries):
        text = alpaca_prompt.format(dialogue, summary) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'dialogue', 'summary', 'text'],
    num_rows: 5000
})


In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.546813
2,2.387688
3,2.464953
4,2.587748
5,2.231201
6,2.450100
7,2.252820
8,2.374238
9,2.194339
10,2.135786


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


# Inference

The fine-tuned model is tested on several new conversations.


In [8]:
FastLanguageModel.for_inference(model)

test_conversations = [
    """John: Are we still meeting at 3 PM?
Sarah: Yes, I'll be at the library.
John: Great. I'll bring the documents.""",

    """Tom: Did you finish the project report?
Mike: Almost. I still need to add the results section.
Tom: Okay, send it to me when you're done.""",

    """Alice: Are you coming to the party tonight?
Bob: I don't think so. I have an exam tomorrow.
Alice: No problem. Good luck with your exam!""",

    """David: Can you pick up some groceries?
Emma: Sure. What do we need?
David: Milk, bread and eggs.
Emma: I'll get them on my way home."""
]

for conversation in test_conversations:
    print("=" * 80)
    print("CONVERSATION:")
    print(conversation)
    print("=" * 80)

    prompt = alpaca_prompt.format(conversation, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        use_cache=True,
        do_sample=False,
    )

    answer = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("MODEL OUTPUT:")
    print(answer)
    print()


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONVERSATION:
John: Are we still meeting at 3 PM?
Sarah: Yes, I'll be at the library.
John: Great. I'll bring the documents.


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL OUTPUT:
Below is a conversation between multiple people.

### Conversation:
John: Are we still meeting at 3 PM?
Sarah: Yes, I'll be at the library.
John: Great. I'll bring the documents.

### Summary:
John and Sarah are going to meet at 3 PM at the library. John will bring the documents.

CONVERSATION:
Tom: Did you finish the project report?
Mike: Almost. I still need to add the results section.
Tom: Okay, send it to me when you're done.


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL OUTPUT:
Below is a conversation between multiple people.

### Conversation:
Tom: Did you finish the project report?
Mike: Almost. I still need to add the results section.
Tom: Okay, send it to me when you're done.

### Summary:
Mike will send Tom his project report when he's finished with it.

CONVERSATION:
Alice: Are you coming to the party tonight?
Bob: I don't think so. I have an exam tomorrow.
Alice: No problem. Good luck with your exam!


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL OUTPUT:
Below is a conversation between multiple people.

### Conversation:
Alice: Are you coming to the party tonight?
Bob: I don't think so. I have an exam tomorrow.
Alice: No problem. Good luck with your exam!

### Summary:
Bob won't come to Alice's party because he has an exam tomorrow.

CONVERSATION:
David: Can you pick up some groceries?
Emma: Sure. What do we need?
David: Milk, bread and eggs.
Emma: I'll get them on my way home.
MODEL OUTPUT:
Below is a conversation between multiple people.

### Conversation:
David: Can you pick up some groceries?
Emma: Sure. What do we need?
David: Milk, bread and eggs.
Emma: I'll get them on my way home.

### Summary:
Emma will buy milk, bread and eggs on her way home.



In [9]:
model.save_pretrained("samsum_lora_model")
tokenizer.save_pretrained("samsum_lora_model")
print("LoRA adapter saved to samsum_lora_model")


Unsloth: Restored added_tokens_decoder metadata in samsum_lora_model/tokenizer_config.json.


LoRA adapter saved to samsum_lora_model


# Documentation Notes

- Base model: Qwen2.5-7B-Instruct
- Fine-tuning method: LoRA with 4-bit quantization (QLoRA setup)
- Dataset: SAMSum
- Training subset: 5,000 examples
- Task: dialogue summarization
- Initial training run: 60 steps
- Evaluation: multiple unseen example conversations
